# Exploratory Data Analysis - RTIG Roads Dataset

This notebook loads and analyzes the RTIG roads dataset downloaded via ArcGIS REST API.

In [12]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import polars as pl
import folium
from folium import plugins
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Setup
ROOT = Path('../').resolve()
sys.path.append(str(ROOT))

DATA_RAW = ROOT / 'data' / 'raw' / 'carreteras_RTIG.geojson'
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f'Raw data path: {DATA_RAW}')
print(f'File exists: {DATA_RAW.exists()}')
print(f'File size: {DATA_RAW.stat().st_size / 1e6:.2f} MB')

Raw data path: data/raw/carreteras_RTIG.geojson
File exists: True
File size: 167.41 MB


## 1. Load Dataset

In [14]:
# Load processed GeoJSON with geopandas
try:
    # Try loading processed version first (from pipeline)
    processed_gdf = DATA_PROCESSED / 'roads_processed_gdf.parquet'
    if processed_gdf.exists():
        gdf = gpd.read_parquet(processed_gdf)
        print(f'Loaded {len(gdf)} features from processed data')
    else:
        # Fallback to raw (will require custom parsing)
        gdf = gpd.read_file(DATA_RAW)
        print(f'Loaded {len(gdf)} features from raw data')
except Exception as e:
    print(f'Error loading data: {e}')
    gdf = gpd.GeoDataFrame()

print(f'CRS: {gdf.crs}')
print(f'\nGeometry types:')
print(gdf.geometry.type.value_counts())

Loaded 1602 features from processed data
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "ea

## 2. Column Overview

In [15]:
print('Columns and types:')
print(gdf.dtypes)
print('\n' + '='*60)
print(f'Total columns: {len(gdf.columns)}')
print(f'Non-geometry columns: {len(gdf.columns) - 1}')

Columns and types:
id                     int64
carretera             object
longitud               int64
pk_inicio            float64
pk_fin               float64
tipo_de_via           object
valido_desde           int64
valido_hasta          object
titular               object
tent                  object
tent_red_basica       object
tent_corredor         object
geometry            geometry
length_km            float64
min_lon              float64
max_lon              float64
min_lat              float64
max_lat              float64
center_lon           float64
center_lat           float64
num_vertices           int64
curve_complexity     float64
is_tent                int64
dtype: object

Total columns: 23
Non-geometry columns: 22


## 3. Data Quality Assessment

In [16]:
# Null analysis
null_counts = gdf.isnull().sum()
null_pct = (null_counts / len(gdf) * 100).round(2)

null_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Null_Pct': null_pct.values
})

null_df = null_df[null_df['Null_Count'] > 0].sort_values('Null_Pct', ascending=False)
print('Null value analysis:')
print(null_df.to_string(index=False))
print(f'\nTotal rows with at least one null: {gdf.isnull().any(axis=1).sum()}')

Null value analysis:
         Column  Null_Count  Null_Pct
   valido_hasta        1602    100.00
  tent_corredor        1427     89.08
tent_red_basica        1067     66.60
    tipo_de_via          11      0.69
        titular          10      0.62

Total rows with at least one null: 1602


## 4. Geometry Validity

In [17]:
# Check geometry validity
gdf['geom_valid'] = gdf.geometry.is_valid
gdf['geom_empty'] = gdf.geometry.is_empty

print(f'Valid geometries: {gdf["geom_valid"].sum()} / {len(gdf)}')
print(f'Empty geometries: {gdf["geom_empty"].sum()} / {len(gdf)}')

# Compute geometry metrics in a projected CRS so distances remain meaningful.
gdf_metric = gdf.to_crs('EPSG:3857')
gdf['geom_length_km_check'] = gdf_metric.geometry.length / 1000

print(f'\nGeometry length stats (km):')
print(gdf['geom_length_km_check'].describe())

Valid geometries: 1602 / 1602
Empty geometries: 0 / 1602

Geometry length stats (km):
count    1602.000000
mean       24.050371
std        38.864919
min         0.000140
25%         2.205911
50%         6.747051
75%        25.617186
max       303.812577
Name: geom_length, dtype: float64


/var/folders/sd/2d4k328s77l4gfj8jz9z3n340000gn/T/ipykernel_46470/1406941639.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['geom_length'] = gdf.geometry.length


## 5. Key Attribute Statistics

In [18]:
# Display first few rows to inspect attribute structure
print('First 3 rows (non-geometry):' )
display(gdf.iloc[:3, :-1])

First 3 rows (non-geometry):


,id,carretera,longitud,pk_inicio,pk_fin,tipo_de_via,valido_desde,valido_hasta,titular,tent,...,max_lon,min_lat,max_lat,center_lon,center_lat,num_vertices,curve_complexity,is_tent,geom_valid,geom_empty
0,1,A-66R,1119,149.430,150.375,Carretera convencional,1761523200000,None,Estado,NO,...,-621841.128238,5.239603e+06,5.240705e+06,-622314.850632,5.240154e+06,44,27.183071,1,True,False
1,2,A-66R,916,152.340,152.960,Carretera convencional,1761523200000,None,Estado,NO,...,-621465.879959,5.236836e+06,5.236917e+06,-622084.381223,5.236876e+06,9,6.712943,1,True,False
2,3,A-66R,1633,182.375,184.015,Carretera convencional,1761523200000,None,Estado,NO,...,-620237.757200,5.197680e+06,5.198058e+06,-621308.683749,5.197869e+06,28,12.158298,1,True,False


In [19]:
# Numeric summary for numeric columns only
numeric_cols = gdf.select_dtypes(include=['number']).columns.tolist()
print(f'Numeric columns: {numeric_cols}')
print('\nNumeric summary:')
print(gdf[numeric_cols].describe().round(3))

Numeric columns: ['id', 'longitud', 'pk_inicio', 'pk_fin', 'valido_desde', 'length_km', 'min_lon', 'max_lon', 'min_lat', 'max_lat', 'center_lon', 'center_lat', 'num_vertices', 'curve_complexity', 'is_tent', 'geom_length']

Numeric summary:
             id    longitud  pk_inicio    pk_fin  valido_desde  length_km  \
count  1602.000    1602.000   1602.000  1602.000  1.602000e+03   1602.000   
mean    801.500   18250.835    200.273   218.518  1.761523e+12     24.050   
std     462.602   29607.525    248.819   250.498  0.000000e+00     38.865   
min       1.000       0.000     -0.370     0.040  1.761523e+12      0.000   
25%     401.250    1680.500      4.910    20.032  1.761523e+12      2.206   
50%     801.500    5126.000    105.300   129.055  1.761523e+12      6.747   
75%    1201.750   19654.750    316.740   336.162  1.761523e+12     25.617   
max    1602.000  232827.000   1240.955  1243.870  1.761523e+12    303.813   

           min_lon      max_lon      min_lat      max_lat   center

## 6. Categorical & String Attributes

In [20]:
# Object (string) columns
string_cols = gdf.select_dtypes(include=['object']).columns.tolist()
# Remove geometry column
string_cols = [c for c in string_cols if c != 'geometry']

print(f'String/Object columns: {string_cols}')
print()

for col in string_cols:
    unique_count = gdf[col].nunique()
    print(f'\n{col}: {unique_count} unique values')
    if unique_count <= 20:
        print(gdf[col].value_counts().head(10))

String/Object columns: ['carretera', 'tipo_de_via', 'valido_hasta', 'titular', 'tent', 'tent_red_basica', 'tent_corredor']


carretera: 458 unique values

tipo_de_via: 4 unique values
tipo_de_via
Carretera convencional     789
Autopista libre\Autovía    534
Multicarril                189
Autopista peaje             79
Name: count, dtype: int64

valido_hasta: 0 unique values
Series([], Name: count, dtype: int64)

titular: 17 unique values
titular
Estado                          1442
Comunidad Foral de Navarra        32
Diputación Foral de Gipuzkoa      22
Diputación Foral de Álava         19
Generalitat de Catalunya          19
Diputación Foral de Bizkaia       17
Junta de Andalucía                11
Consell de Mallorca                7
Cabildo de Tenerife                7
Junta de Extremadura               4
Name: count, dtype: int64

tent: 2 unique values
tent
NO    1067
SI     535
Name: count, dtype: int64

tent_red_basica: 2 unique values
tent_red_basica
Comprehensive    304
Core   

## 7. Geographic Distribution

In [21]:
# Get bounds
bounds = gdf.total_bounds
print(f'Geographic bounds:')
print(f'  Min X: {bounds[0]:.4f}, Max X: {bounds[2]:.4f}')
print(f'  Min Y: {bounds[1]:.4f}, Max Y: {bounds[3]:.4f}')

# Approximate center
center_x = (bounds[0] + bounds[2]) / 2
center_y = (bounds[1] + bounds[3]) / 2
print(f'\nApprox center: ({center_y:.4f}, {center_x:.4f})')

Geographic bounds:
  Min X: -1860182.8541, Max X: 352632.4540
  Min Y: 3218321.4656, Max Y: 5417723.7174

Approx center: (4318022.5915, -753775.2001)


## 8. Create Base Map Visualization

In [23]:
# Create a folium map centered on the dataset
m = folium.Map(
    location=[center_y, center_x],
    zoom_start=6,
    tiles='OpenStreetMap'
)

# Add road segments (sample for performance)
sample_gdf = gdf.sample(min(500, len(gdf)), random_state=42)

for idx, row in sample_gdf.iterrows():
    if row.geometry.geom_type == 'LineString':
        coords = [[point[1], point[0]] for point in row.geometry.coords]
        folium.PolyLine(
            coords,
            color='blue',
            weight=2,
            opacity=0.3
        ).add_to(m)

# Create maps directory if it doesn't exist
maps_dir = ROOT / 'maps'
maps_dir.mkdir(parents=True, exist_ok=True)

map_path = maps_dir / 'base_map.html'
m.save(str(map_path))
print(f'Base map saved to {map_path}')

Base map saved to maps/base_map.html


## 9. Summary & Key Findings

In [24]:
summary = f"""
=== RTIG ROADS DATA - EDA SUMMARY ===

Dataset Overview:
  - Total records: {len(gdf)}
  - Total columns: {len(gdf.columns)}
  - Geometry type(s): {gdf.geometry.type.unique().tolist()}

Data Quality:
  - Valid geometries: {(gdf['geom_valid'].sum() / len(gdf) * 100):.1f}%
  - Rows with nulls: {gdf.isnull().any(axis=1).sum()} ({gdf.isnull().any(axis=1).sum() / len(gdf) * 100:.1f}%)

Geographic Coverage:
  - CRS: {gdf.crs}
  - Longitude range: {bounds[0]:.2f} to {bounds[2]:.2f}
  - Latitude range: {bounds[1]:.2f} to {bounds[3]:.2f}

Next Steps:
  1. Clean data: handle nulls, invalid geometries
  2. Feature engineering: derive road characteristics
  3. Integrate with external data: weather, energy, population
  4. Build scoring model: risk/priority assessment
  5. Create dashboards: map visualizations + KPIs
"""

print(summary)


=== RTIG ROADS DATA - EDA SUMMARY ===

Dataset Overview:
  - Total records: 1602
  - Total columns: 26
  - Geometry type(s): ['LineString']

Data Quality:
  - Valid geometries: 100.0%
  - Rows with nulls: 1602 (100.0%)

Geographic Coverage:
  - CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_syst

In [25]:
# Save condensed version to parquet for later
gdf.to_parquet(DATA_PROCESSED / 'roads_raw_gdf.parquet')
print(f'Raw GeoDataFrame saved to {DATA_PROCESSED / "roads_raw_gdf.parquet"}')

Raw GeoDataFrame saved to data/processed/roads_raw_gdf.parquet
